<a href="https://colab.research.google.com/github/AatiqahHarmine/Assignment8_SSD/blob/main/Assignment8_SSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import unittest
from typing import List, Dict, Optional, Tuple

# --- 1. Grade and Grade Point Mapping ---

class Grade:
    """
    Represents a grade earned in a specific course.
    Also handles the conversion of letter grades to numeric quality points (4.0 scale).
    """

    # Define a mapping for standard letter grades to quality points
    GRADE_POINTS: Dict[str, float] = {
        "A": 4.0,
        "B": 3.0,
        "C": 2.0,
        "D": 1.0,
        "F": 0.0,
    }

    def __init__(self, course: 'Course', letter_grade: str):
        """
        Initializes a Grade object.

        Args:
            course: The Course object this grade is for.
            letter_grade: The letter grade earned (e.g., "A", "B").

        Raises:
            ValueError: If the provided letter grade is not valid.
        """
        self.course = course
        self.letter_grade = letter_grade.upper()  # Ensure consistency

        if self.letter_grade not in self.GRADE_POINTS:
            raise ValueError(f"Invalid grade: {letter_grade}. Must be one of {list(self.GRADE_POINTS.keys())}")

        self.quality_points = self.GRADE_POINTS[self.letter_grade]

# --- 2. Course Class ---

class Course:
    """
    Represents a course with a code, title, and defined credit hours.
    Updated to include both a code and a descriptive title, aligning with external calls.
    """
    def __init__(self, code: str, title: str, credits: float):
        """
        Initializes a Course object.

        Args:
            code: The short course code (e.g., "CS101").
            title: The full name of the course (e.g., "Calculus I").
            credits: The number of credit hours for the course.
        """
        if credits <= 0:
            raise ValueError("Course credits must be positive.")
        self.code = code
        self.title = title
        self.credits = credits

# --- 3. Student Class ---

class Student:
    """
    Represents a student who enrolls in courses and holds a record of grades.
    Provides methods for adding grades and calculating GPA.
    """
    def __init__(self, student_id: str, name: str):
        self.student_id = student_id
        self.name = name
        # grades_record stores a mapping of Course objects to Grade objects
        self.grades_record: Dict[Course, Grade] = {}

    def add_grade(self, course: Course, letter_grade: str) -> None:
        """
        Adds a new grade record for a student in a specific course.

        Args:
            course: The Course object.
            letter_grade: The letter grade earned.
        """
        # Create the Grade object, which automatically validates the letter_grade
        grade = Grade(course, letter_grade)
        self.grades_record[course] = grade

    def calculate_gpa(self) -> float:
        """
        Calculates the student's Grade Point Average (GPA).

        Formula: (Total Quality Points * Credit Hours) / (Total Credit Hours)

        Returns:
            The calculated GPA, rounded to two decimal places.
        """
        total_quality_points = 0.0
        total_credits = 0.0

        if not self.grades_record:
            # Required functionality: Students with no courses return GPA as 0.
            return 0.0

        for course, grade in self.grades_record.items():
            # Quality Points for the grade * Credit hours for the course
            points_for_course = grade.quality_points * course.credits

            total_quality_points += points_for_course
            total_credits += course.credits

        if total_credits == 0:
            # Edge case: Should not happen if Course credits validation is correct,
            # but necessary for robustness.
            return 0.0

        raw_gpa = total_quality_points / total_credits
        return round(raw_gpa, 2)

    def generate_transcript(self) -> List[Tuple[str, str, float]]:
        """
        Generates a simple transcript list (Course Title, Letter Grade, Credits).
        Updated to use course.title instead of course.name.
        """
        transcript = []
        for course, grade in self.grades_record.items():
            transcript.append((course.title, grade.letter_grade, course.credits))
        return transcript


# --- 4. Unit Tests ---

class TestGPASystem(unittest.TestCase):
    """
    Unit tests for the Course, Grade, and Student classes.
    """

    def setUp(self):
        """Set up common variables for tests. Course calls updated to match new structure."""
        self.student = Student("S1001", "Alice Smith")

        # Define courses (code, title, credits)
        self.c_calc = Course("MATH101", "Calculus I", 4.0)
        self.c_hist = Course("HIST205", "World History", 3.0)
        self.c_chem = Course("CHEM101L", "Chemistry Lab", 1.0)
        self.c_pe = Course("PE100", "Physical Education", 2.0)

        # Add a set of grades for calculation tests:
        # Calc (4 credits, A=4.0) -> 16.0 points
        # Hist (3 credits, B=3.0) -> 9.0 points
        # Chem (1 credit, C=2.0) -> 2.0 points
        # PE (2 credits, F=0.0)  -> 0.0 points
        self.student.add_grade(self.c_calc, "A")
        self.student.add_grade(self.c_hist, "B")
        self.student.add_grade(self.c_chem, "C")
        self.student.add_grade(self.c_pe, "F")

    def test_gpa_calculation_accuracy(self):
        """
        1. Verify GPA calculation is accurate based on weighted average.

        Expected Calculation:
        Total Points = (4.0 * 4.0) + (3.0 * 3.0) + (2.0 * 1.0) + (0.0 * 2.0)
                     = 16.0 + 9.0 + 2.0 + 0.0 = 27.0
        Total Credits = 4.0 + 3.0 + 1.0 + 2.0 = 10.0
        GPA = 27.0 / 10.0 = 2.70
        """
        expected_gpa = 2.70
        actual_gpa = self.student.calculate_gpa()

        self.assertAlmostEqual(actual_gpa, expected_gpa, 2,
                                "GPA calculation failed to match the expected weighted average.")

    def test_student_no_courses_gpa(self):
        """
        3. Verify students with no courses return GPA as 0.
        """
        empty_student = Student("S1002", "Bob Johnson")
        self.assertEqual(empty_student.calculate_gpa(), 0.0,
                         "GPA for a student with no grades should be 0.0.")

    def test_invalid_grade_rejection(self):
        """
        2. Verify invalid grade values (like 'A+' or 'G') are rejected with a ValueError.
        """
        new_student = Student("S1003", "Charlie Brown")
        new_course = Course("PSYC101", "Psychology", 3.0)

        # Test rejection of a non-standard letter grade ('A+')
        with self.assertRaisesRegex(ValueError, "Invalid grade: A\\+",
                                    msg="Did not reject the grade 'A+'"):
            new_student.add_grade(new_course, "A+")

        # Test rejection of a completely invalid letter grade ('G')
        with self.assertRaisesRegex(ValueError, "Invalid grade: G",
                                    msg="Did not reject the grade 'G'"):
            new_student.add_grade(new_course, "G")

    def test_grade_update(self):
        """
        Verify that adding a grade for the same course overwrites the previous grade.
        """
        test_student = Student("S1004", "Diana Prince")
        test_course = Course("ENG201", "Literature", 3.0)

        # Initial grade: D (1.0)
        test_student.add_grade(test_course, "D")
        self.assertEqual(test_student.calculate_gpa(), 1.00, "Initial GPA should be 1.0.")

        # Update grade: A (4.0)
        test_student.add_grade(test_course, "A")
        self.assertEqual(test_student.calculate_gpa(), 4.00, "Updated GPA should be 4.0.")

    def test_case_insensitivity(self):
        """
        Verify that grade input is case-insensitive.
        """
        test_student = Student("S1005", "Evan Clark")
        test_course = Course("ART105", "Art History", 3.0)

        test_student.add_grade(test_course, "a") # Lowercase 'a'
        self.assertEqual(test_student.calculate_gpa(), 4.00, "Grade system is not case-insensitive.")

    def test_course_credit_validation(self):
        """
        Verify that a Course cannot be initialized with zero or negative credits.
        """
        with self.assertRaises(ValueError, msg="Allowed negative credits for a course."):
            Course("BAD101", "Dummy Course", -1.0)

        with self.assertRaises(ValueError, msg="Allowed zero credits for a course."):
            Course("BAD102", "Dummy Course", 0.0)

# Run the tests
if __name__ == '__main__':
    # This runs the defined unit tests automatically
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

    # Example usage:
    print("\n--- Example System Usage ---")

    # 1. Create student and courses (code, title, credits)
    student_1 = Student("9001", "Jane Doe")
    course_math = Course("MATH300", "Discrete Math", 3.0)
    course_prog = Course("CS101", "Intro Programming", 4.0)
    course_art = Course("ART300", "Sculpture 101", 3.0)

    # 2. Add grades
    student_1.add_grade(course_math, "B") # 3.0 points * 3 credits = 9.0
    student_1.add_grade(course_prog, "A") # 4.0 points * 4 credits = 16.0
    student_1.add_grade(course_art, "C")  # 2.0 points * 3 credits = 6.0

    # 3. Calculate GPA
    # Total Points = 9.0 + 16.0 + 6.0 = 31.0
    # Total Credits = 3.0 + 4.0 + 3.0 = 10.0
    # Expected GPA = 31.0 / 10.0 = 3.10
    gpa = student_1.calculate_gpa()

    print(f"Student: {student_1.name} (ID: {student_1.student_id})")
    print("-" * 30)
    print(f"Calculated GPA: {gpa}")
    print(f"Transcript:")

    for name, grade, credits in student_1.generate_transcript():
        print(f"  - {name} ({credits} cr): {grade}")


.......F...
FAIL: test_gpa_calculation (__main__.TestGPA_Calculation.test_gpa_calculation)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/tmp/ipython-input-1381733127.py", line 15, in test_gpa_calculation
    self.assertEqual(self.student.calculate_gpa(), 8.86)
AssertionError: 3.43 != 8.86

----------------------------------------------------------------------
Ran 11 tests in 0.009s

FAILED (failures=1)



--- Example System Usage ---
Student: Jane Doe (ID: 9001)
------------------------------
Calculated GPA: 3.1
Transcript:
  - Discrete Math (3.0 cr): B
  - Intro Programming (4.0 cr): A
  - Sculpture 101 (3.0 cr): C
